# SpecMine v1.0 sample — worked examples

A short tour of the 500-repository sample: the tool census, spec anatomy, the
spec→code PR layer, and the traceability index. Everything runs on the Parquet
files in `data/` and `specs.jsonl.gz`.

```bash
pip install pandas pyarrow matplotlib
```

In [1]:
import gzip, json
import pandas as pd

spec_files = pd.read_parquet('data/spec_files.parquet')
features   = pd.read_parquet('data/spec_content_features.parquet')
prs        = pd.read_parquet('data/pull_requests.parquet')
pr_files   = pd.read_parquet('data/pr_files.parquet')
links      = pd.read_parquet('data/spec_links.parquet')
kiro       = pd.read_parquet('data/kiro_files.parquet')
print(f'{len(spec_files):,} specs across {spec_files.repo_name.nunique():,} repos')

28,583 specs across 415 repos


## 1. Which tools produced the specs?

`spec_tool` is the path-fingerprint attribution. Named tools and interpretable
path buckets sit in the same column.

In [2]:
spec_files.groupby('spec_tool').agg(
    files=('file_url_sha16', 'size'),
    repos=('repo_name', 'nunique')).sort_values('files', ascending=False)

Out[2]: 
               files  repos
spec_tool                  
openspec       22132    147
speckit         1863     85
docs_embedded   1500     49
specweave        962      3
moai             648      3
specs_generic    484     36
codex            266      3
kitty_specs      260      3
adhoc_other      253     90
conductor        116      5
dot_specs         37      3
claude_code       14     12
root_bare         13     13
trae              10      3
agent_os           5      3
gsd                4      3
opencode           4      3
kilocode           3      3
cursor             3      3
caffeine           3      3
zenflow            3      3


## 2. Adoption over time

Each repository's earliest spec commit dates when it adopted SDD.

In [3]:
first = (spec_files[spec_files.file_first_commit_at != '']
         .groupby('repo_name').file_first_commit_at.min())
first = pd.to_datetime(first, errors='coerce').dropna()
first.dt.to_period('Q').value_counts().sort_index()

Out[3]: 
file_first_commit_at
2024Q1      2
2024Q2      4
2024Q3      1
2024Q4      4
2025Q1      4
2025Q2      4
2025Q3     11
2025Q4     56
2026Q1    164
2026Q2    159
2026Q3      6
Freq: Q-DEC, Name: count, dtype: int64


<ipython-input-3-7803d3a68a5e>:4: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  first.dt.to_period('Q').value_counts().sort_index()


## 3. How big are specs, and what do they contain?

`spec_content_features` has 39 columns per spec (size, structure, requirement
syntax, section markers, quality signals).

In [4]:
print('median spec length (lines):', int(features.n_lines.median()))
marker_cols = [c for c in features.columns if c.startswith('has_')]
(features[marker_cols].mean().sort_values(ascending=False) * 100).round(1)

median spec length (lines): 56
Out[4]: 
has_requirements            87.8
has_gherkin                 82.0
has_ears                    71.5
has_purpose                 16.4
has_overview                 9.9
has_user_scenarios           9.0
has_user_story               8.4
has_out_of_scope             7.7
has_success_criteria         7.3
has_unfilled_placeholder     7.0
has_dependencies             6.7
has_edge_cases               5.5
has_mandatory_marker         3.5
has_acceptance_criteria      3.2
has_non_goals                2.5
has_open_questions           1.1
has_lorem                    0.0
dtype: float64


## 4. The flagship question: which specs change in the same PR as code?

`touches_code = 1` marks a PR that edits a spec and also changes source files
outside the spec directory. This is the co-change signal the PR layer is built on.

In [5]:
cochange = prs[prs.touches_code == 1]
print(f'{len(cochange):,} of {len(prs):,} spec-touching PRs also change code '
      f'({100*len(cochange)/len(prs):.1f}%)')

key = ['tool', 'repo_name', 'pr_number']
changed = pr_files.merge(cochange[key], on=key)
print('spec files changed:', int(changed.is_spec.sum()),
      '| code files changed:', int(changed.is_code.sum()))
cochange.assign(size=cochange.additions + cochange.deletions)[
    ['tool', 'repo_name', 'pr_number', 'n_spec_files', 'n_code_files', 'size']
].sort_values('size', ascending=False).head()

4,294 of 5,335 spec-touching PRs also change code (80.5%)
spec files changed: 22481 | code files changed: 143857
Out[5]: 
          tool                        repo_name  ...  n_code_files    size
3820   speckit             complytime/complyctl  ...          2488  749516
4214   speckit              davidmatousek/tachi  ...           174  732714
4255   speckit              davidmatousek/tachi  ...           175  713716
1452  openspec          carverauto/serviceradar  ...          2476  632039
1022  openspec  deepcausality-rs/deep_causality  ...          1579  555009

[5 rows x 6 columns]


## 5. The traceability index: a second channel from spec to code

`spec_links` records typed references a spec makes to code files, sibling docs,
PRs, issues, and branches, independent of whether they co-change.

In [6]:
display(links.rel.value_counts())
display(links.provenance.value_counts())

rel
code       105859
sibling     68675
pr          64150
ref         13753
issue        7367
branch       1224
anchor          4
Name: count, dtype: int64

provenance
tasks            105256
commit_msg        84830
tree              69029
branch_header      1224
content             693
Name: count, dtype: int64

## 6. Reading raw spec text

`specs.jsonl.gz` holds each spec's Markdown, joined on `file_url_sha16`.

In [7]:
with gzip.open('specs.jsonl.gz', 'rt', encoding='utf-8') as f:
    for line in f:
        rec = json.loads(line)
        if rec['spec_tool'] == 'speckit' and len(rec['content']) > 800:
            print(rec['repo_name'], '::', rec['file_path'])
            print(rec['content'][:800])
            break

smart-mcp-proxy/mcpproxy-go :: specs/073-activity-size-retention/spec.md
# Feature Specification: Activity-Log Size-Based Retention Cap

**Feature Branch**: `073-activity-size-retention`
**Created**: 2026-06-04
**Status**: Draft
**Input**: Bound the local database's unbounded growth from the activity log by adding a total-size retention cap, complementing the existing age- and count-based caps.

## Why (Problem)

The local activity log (a record of every tool call, policy decision, and quarantine change) is retained only by **age** (default 90 days) and **record count** (default 100,000). Because each record can carry a large request/response payload (truncated at 64 KB), the log can grow to **hundreds of megabytes while staying well under both caps** — on a real instance it reached ~438 MB across ~93,000 records, the single largest contributor to a ~1 GB local


In [8]:
import pandas as pd

specs = pd.read_parquet("data/spec_files.parquet")

print("Rows:", len(specs))
pd.set_option("display.max_colwidth", None)
display(specs.head(10))
print(specs.columns.tolist())

Rows: 28583
['file_url_sha16', 'file_url', 'file_path', 'file_name', 'file_sha', 'git_url', 'search_run_id', 'repo_id', 'repo_node_id', 'repo_name', 'repo_url', 'repo_homepage', 'repo_description', 'repo_fork', 'repo_private', 'repo_visibility', 'owner_login', 'owner_id', 'owner_node_id', 'owner_type', 'owner_html_url', 'owner_avatar_url', 'owner_site_admin', 'repo_stars', 'repo_forks', 'repo_watchers', 'repo_subscribers_count', 'repo_network_count', 'repo_open_issues', 'repo_size_kb', 'repo_language', 'repo_license', 'repo_license_key', 'repo_license_spdx_id', 'repo_default_branch', 'repo_topics', 'repo_archived', 'repo_disabled', 'repo_is_template', 'repo_allow_forking', 'repo_web_commit_signoff_required', 'repo_has_issues', 'repo_has_projects', 'repo_has_wiki', 'repo_has_pages', 'repo_has_downloads', 'repo_has_discussions', 'repo_clone_url', 'repo_ssh_url', 'repo_svn_url', 'repo_git_url', 'repo_mirror_url', 'repo_created_at', 'repo_updated_at', 'repo_pushed_at', 'file_last_commit_sh

,file_url_sha16,file_url,file_path,file_name,file_sha,git_url,search_run_id,repo_id,repo_node_id,repo_name,repo_url,repo_homepage,repo_description,repo_fork,repo_private,repo_visibility,owner_login,owner_id,owner_node_id,owner_type,owner_html_url,owner_avatar_url,owner_site_admin,repo_stars,repo_forks,repo_watchers,repo_subscribers_count,repo_network_count,repo_open_issues,repo_size_kb,repo_language,repo_license,repo_license_key,repo_license_spdx_id,repo_default_branch,repo_topics,repo_archived,repo_disabled,repo_is_template,repo_allow_forking,...,file_first_commit_at,file_first_commit_author,file_first_commit_author_login,file_first_commit_author_id,file_first_commit_author_type,file_first_commit_author_html_url,file_first_commit_author_avatar_url,file_first_commit_author_site_admin,file_first_commit_committer,file_first_commit_committer_at,file_first_commit_committer_login,file_first_commit_committer_id,file_first_commit_committer_type,file_first_commit_committer_html_url,file_first_commit_committer_avatar_url,file_first_commit_committer_site_admin,file_first_commit_tree_sha,file_first_commit_parents,file_first_commit_verified,file_first_commit_verification_reason,file_first_commit_verified_at,file_total_commits,commit_history_status,commit_history_fetched_at,spec_tool,spec_role,file_content_sha,file_content_bytes,content_fetch_status,content_fetched_at,feature_branch,n_linked_prs,n_linked_issues,n_sibling_docs,n_code_links,has_tasks,has_plan,has_proposal,has_design,trace_rollup_at
0,00073b579da0f72d,https://github.com/igamenovoer/houmao/blob/0d0a4cd3f4888f786c21f6b1a0a8893332805bce/openspec/changes/archive/2026-05-14-retire-legacy-loop-skills/specs/docs-loop-authoring-guide/spec.md,openspec/changes/archive/2026-05-14-retire-legacy-loop-skills/specs/docs-loop-authoring-guide/spec.md,spec.md,5c51930fd758bcdf82f374b5dff97d7444592844,https://api.github.com/repositories/1173411389/git/blobs/5c51930fd758bcdf82f374b5dff97d7444592844,spec_2024_2026,1173411389,R_kgDORfDWPQ,igamenovoer/houmao,https://github.com/igamenovoer/houmao,None,A framework and CLI toolkit for orchestrating teams of loosely-coupled AI agents.,0,0,public,igamenovoer,9019398,MDQ6VXNlcjkwMTkzOTg=,User,https://github.com/igamenovoer,https://avatars.githubusercontent.com/u/9019398?v=4,0,17,2,17,1,2,11,46215,Python,Apache License 2.0,apache-2.0,Apache-2.0,main,,0,0,0,1,...,2026-05-14T17:58:47Z,igamenovoer,igamenovoer,9019398.0,User,https://github.com/igamenovoer,https://avatars.githubusercontent.com/u/9019398?v=4,0.0,igamenovoer,2026-05-14T17:58:47Z,igamenovoer,9019398.0,User,https://github.com/igamenovoer,https://avatars.githubusercontent.com/u/9019398?v=4,0.0,4f5a108e939ddfa3221ddb4efe9d1c2fbbe3fef2,b795e0b16fa0c9ec0089a03df357604947588a96,0,unsigned,None,1,done,2026-07-11T00:23:58Z,openspec,archived,a16020906e2a517de6e2cd453b2653be39a091e53217726507e29a6a11f563ca,1189,done,2026-07-12T01:10:54Z,None,0,0,3,0,1,0,1,1,2026-07-21T17:18:38Z
1,000c8bf235eb04c7,https://github.com/smart-mcp-proxy/mcpproxy-go/blob/b26b68556c36071f7679a50cb30e6e4fde8b9e1c/specs/073-activity-size-retention/spec.md,specs/073-activity-size-retention/spec.md,spec.md,4a0407799f96d6a1483db8326591c552908db9a7,https://api.github.com/repositories/1004056914/git/blobs/4a0407799f96d6a1483db8326591c552908db9a7,spec_2024_2026,1004056914,R_kgDOO9ixUg,smart-mcp-proxy/mcpproxy-go,https://github.com/smart-mcp-proxy/mcpproxy-go,https://mcpproxy.app,"Supercharge AI Agents, Safely",0,0,public,smart-mcp-proxy,216428285,O_kgDODOZu_Q,Organization,https://github.com/smart-mcp-proxy,https://avatars.githubusercontent.com/u/216428285?v=4,0,276,35,276,3,35,12,208673,Go,MIT License,mit,MIT,main,"ai, ai-agents, audit-logging, bm25, cli, context-window, developer-tools, docker, golang, llm, llm-tools, local-first, mcp, mcp-proxy, mcp-server, model-context-protocol, proxy-server, security, tool-routing, web-ui",0,0,0,1,...,2026-06-04T04:07:43Z,Dumbris,Dumbris,1362215.0,User,https://github.com/Dumbris,https://avatars.github

## 7. Manual rubric pilot: collect specs to read and grade

**Run all cells from the repository directory.** This section uses local data only and preserves the earlier examples. It produces 20 ungraded candidates, complete reading copies, and independent reviewer worksheets in `output/rubric_pilot/`. Adjust the settings below to draw more candidates.

### Planning basis and scope
- [CS Project Deliverables](../CS%20Project%20Deliverables.pdf), rubric notes: identify **5 good and 5 bad** examples by reading them and explaining the labels. These are targets after human review, not labels assigned by the sampler.
- [Detailed project plan](../CS520_Project_Plan_VS.pdf): pilot 10 specs; use five 0–2 dimensions; have two reviewers score independently without outcomes; reconcile, freeze the rubric, and rescore the pilot.
- [Final project plan](../CS520_Final_Project_Plan_MSR2027.pdf) and [initial project plan](../CS520%20Project%20Plan.pdf): compare similar scopes and assess completeness, clarity, conciseness, references, and validation. Length alone is not quality.
- The documents contain different RQ numbering. This section implements their shared rubric-development task only.

This is a **purposive calibration sample**, not a representative estimate or a validated scorer. We prioritize Spec Kit feature specs and OpenSpec change/archive specs, consistent with the detailed plan. Kiro designs/tasks and living project-wide specs are outside this initial comparison. Archived content is retained as a candidate, not assumed to be the original pre-implementation text. Final outcome analysis must retrieve and verify the baseline historical revision.

Selection balances tools and length bands, caps repository concentration, and removes exact duplicate text. It does not use quality labels, PR/fix counts, stars, or downstream outcomes. Reviewers still need to classify actual task scope; length bands do not establish comparable scope. Missing text is unavailable, never a zero grade.

In [2]:
from pathlib import Path
from collections import Counter
import hashlib
import html
from IPython.display import HTML, display

SEED = 520
N_CANDIDATES = 20
MAX_PER_REPO = 2
MIN_WORDS, MAX_WORDS = 80, 2500  # Reading-burden limits, NOT quality thresholds.
TOOLS = ('speckit', 'openspec')
assert N_CANDIDATES > 0 and MAX_PER_REPO > 0

pilot_meta = spec_files.merge(features, on='file_url_sha16', validate='one_to_one')
role_ok = ((pilot_meta.spec_tool.eq('speckit') & pilot_meta.spec_role.eq('feature')) |
           (pilot_meta.spec_tool.eq('openspec') & pilot_meta.spec_role.isin(['change_proposal', 'archived'])))
eligible = pilot_meta.loc[
    role_ok & pilot_meta.spec_tool.isin(TOOLS) & pilot_meta.lang.eq('en') &
    pilot_meta.content_fetch_status.eq('done') &
    pilot_meta.n_words.between(MIN_WORDS, MAX_WORDS)
].copy()
# Stable ordering independent of input row order or Python's randomized hash().
eligible['selection_order'] = eligible.file_url_sha16.map(
    lambda value: hashlib.sha256(f'{SEED}:{value}'.encode()).hexdigest())
eligible['length_band'] = pd.cut(eligible.n_words, [0, 300, 800, float('inf')],
                               labels=['short', 'medium', 'long']).astype(str)
wanted = set(eligible.file_url_sha16)
text_by_id = {}
with gzip.open('specs.jsonl.gz', 'rt', encoding='utf-8') as stream:
    for line in stream:
        record = json.loads(line)
        if record['file_url_sha16'] in wanted and isinstance(record.get('content'), str):
            if record['content'].strip():
                text_by_id[record['file_url_sha16']] = record['content']
print(f'{len(eligible):,} eligible metadata rows; {len(text_by_id):,} with readable text.')
eligible = eligible[eligible.file_url_sha16.isin(text_by_id)].copy()
eligible['text_sha256'] = eligible.file_url_sha16.map(
    lambda key: hashlib.sha256(text_by_id[key].encode('utf-8')).hexdigest())
eligible = eligible.sort_values('selection_order').drop_duplicates('text_sha256')

# Rotate across tool × length strata, taking at most MAX_PER_REPO per repository.
queues = [list(eligible[(eligible.spec_tool == tool) & (eligible.length_band == band)].index)
          for band in ['short', 'medium', 'long'] for tool in TOOLS]
chosen, per_repo = [], Counter()
while len(chosen) < N_CANDIDATES:
    before = len(chosen)
    for queue in queues:
        while queue:
            idx = queue.pop(0)
            repo = eligible.loc[idx, 'repo_name']
            if per_repo[repo] < MAX_PER_REPO:
                chosen.append(idx)
                per_repo[repo] += 1
                break
        if len(chosen) == N_CANDIDATES:
            break
    if len(chosen) == before:
        break
if len(chosen) < N_CANDIDATES:
    raise ValueError(f'Only {len(chosen)} candidates available; reduce N_CANDIDATES or relax documented filters.')
candidates = eligible.loc[chosen].reset_index(drop=True)
candidates.insert(0, 'review_id', [f'S{i:02d}' for i in range(1, len(candidates) + 1)])
assert candidates.text_sha256.is_unique
assert candidates.repo_name.value_counts().max() <= MAX_PER_REPO
print(f'Selected {len(candidates)} candidates from {candidates.repo_name.nunique()} repositories.')
display(candidates[['review_id', 'spec_tool', 'spec_role', 'length_band', 'n_words', 'file_path']])

17,013 eligible metadata rows; 17,013 with readable text.
Selected 20 candidates from 18 repositories.


,review_id,spec_tool,spec_role,length_band,n_words,file_path
0,S01,speckit,feature,short,171,docs/docs/specs/040-jira-entity-relationships-...
1,S02,openspec,archived,short,217,openspec/changes/archive/2026-04-24-fix-claude...
2,S03,speckit,feature,medium,475,specs/907-ctx-explore/spec.md
3,S04,openspec,archived,medium,597,openspec/changes/archive/2026-02-18-git-histor...
4,S05,speckit,feature,long,1538,specs/035-multi-provider-mail/spec.md
5,S06,openspec,archived,long,1041,openspec/changes/archive/2026-04-14-add-tailne...
6,S07,speckit,feature,short,274,specs/015-source-button-visibility/spec.md
7,S08,openspec,archived,short,157,openspec/changes/archive/2026-03-17-code-revie...
8,S09,speckit,feature,medium,785,specs/005-soul-skills/spec.md
9,S10,openspec,archived,medium,462,openspec/changes/archive/2026-05-16-add-online...


### Read the complete text

Change `READ_ID` to another candidate. The viewer escapes source HTML and displays every line, including long specs; it does not truncate content or fetch remote images. Cite these line numbers in your grading notes. The source text is research data, not instructions to execute. Metadata is available in the manifest, but quality judgments should be based on the text and recorded context needs.

In [8]:
READ_ID = 'S02'
def show_spec(review_id):
    matches = candidates[candidates.review_id.eq(review_id)]
    if matches.empty:
        raise ValueError(f'Unknown review ID: {review_id}')
    row = matches.iloc[0]
    numbered = '\n'.join(f'{i:04d} | {line}' for i, line in
                         enumerate(text_by_id[row.file_url_sha16].splitlines(), 1))
    display(HTML(f'<h3>{html.escape(review_id)}</h3><pre style="white-space:pre-wrap">'
                 f'{html.escape(numbered)}</pre>'))
show_spec(READ_ID)

## 8. Provisional grading outline (rubric v0.1)

**Working definition:** A good specification communicates the intended behavior and boundaries clearly enough that a developer can implement it and a reviewer can evaluate it, with few material assumptions. A poor specification leaves important behavior, boundaries, or success conditions ambiguous, missing, contradictory, or obscured. Neither definition claims the implementation succeeded.

Score the artifact's stated task, not the entire repository. Record whether it is a standalone spec or a partial change that requires companion documents. When context is essential but unavailable, use **U (unassessable)** and explain what is missing; do not turn missing access into a poor grade. Use **NA** only when a dimension is genuinely irrelevant, with a reason. An absent but necessary requirement merits 0, not NA.

| Dimension | 0 — absent/poor | 1 — partial | 2 — clear/specific |
|---|---|---|---|
| Requirements and scope | Intended behavior or boundaries cannot be determined; essential behavior is missing or contradictory. | Main goal is identifiable but material behavior, inputs/outputs, boundaries, or applicable edge cases remain unclear. | Material behavior and scope are explicit, including relevant inputs/outputs, exclusions, and exceptional cases. |
| Testable acceptance criteria | No observable success condition; only aspirations such as “works well.” | Some observable results, but important conditions or expected results are underspecified. | Material requirements have observable pass/fail conditions, with scenarios or thresholds where relevant. |
| Clarity and lack of redundancy | Ambiguity, contradictions, or filler substantially obstruct interpretation. | Mostly understandable, but some vague terms, repetition, or inconsistent terminology affects interpretation. | Precise, consistent, concise wording; useful detail without distracting duplication. |
| Relevant references and constraints | Necessary interfaces, dependencies, constraints, or context cannot be identified. | Some relevant context is given, but important assumptions or dependencies remain unclear. | Necessary context and constraints are identifiable and explained; references are purposeful. No link-count bonus. |
| Verification guidance | No usable guidance on how to check the intended behavior. | A testing approach is mentioned, but cases, setup, or checks are too vague to reproduce. | Appropriate checks, cases, setup, or validation steps explain how to verify the intended behavior. No named framework required. |

Acceptance criteria ask **what counts as success**; verification guidance asks **how to check it**. They may share evidence, so record overlap rather than inventing separate support. A self-contained spec can earn 2 for references/constraints without external links when it explicitly supplies the relevant context.

### Review procedure
1. Read all candidates and record a one-sentence purpose, task type (feature/fix/refactor/docs/other), scope (small feature/subsystem/project-wide/unclear), and missing context. Do not assume length means scope.
2. Each of two reviewers records an initial `good`, `bad`, `mixed`, or `uncertain` impression **before** applying the rubric, with reasons. Then score the five dimensions independently, quoting line-numbered evidence for every score.
3. Compare independent records. Preserve originals, discuss disagreements, and write a separate consensus record. Prefer similar task/scope groups when choosing contrasting examples.
4. Select **5 good and 5 bad** examples using defensible consensus explanations. Do not force borderline cases into either label to fill the quota; expand the candidate pool if needed. The remaining candidates are reserves, not a held-out validation set once read.
5. Revise ambiguous anchors, record the changes, freeze v1.0, and rescore the selected 10. Validate later on a new sample, preferably from new repositories.

For five numeric scores, the descriptive total is **0–10** (or total × 10 as a percentage). Do not assign high/low cutoffs before the pilot. If any score is U, NA, or blank, leave the full total blank and report the dimension profile. Store scores separately so later work can study individual characteristics.

Report per-dimension exact agreement on jointly numeric ratings, the number of jointly rated cases, and disagreements; do not treat missing values as agreement. A pilot of 10 is rubric development, not proof of predictive validity. Avoid looking at later fixes or PR counts while grading. Snapshot grades cannot answer whether *pre-implementation* quality predicted outcomes.

In [7]:
# Materialize a reproducible reading packet and editable JSON/Markdown worksheets.
# Existing reviewer files are NEVER overwritten on reruns.
DIMENSIONS = ['requirements_scope', 'acceptance_criteria', 'clarity_conciseness',
              'references_constraints', 'verification_guidance']
manifest_cols = ['review_id', 'file_url_sha16', 'repo_name', 'file_path', 'file_url',
                 'spec_tool', 'spec_role', 'n_words', 'length_band', 'text_sha256',
                 'file_content_sha', 'content_fetched_at']
manifest = json.loads(candidates[manifest_cols].to_json(orient='records'))
settings = dict(seed=SEED, count=N_CANDIDATES, max_per_repo=MAX_PER_REPO,
                min_words=MIN_WORDS, max_words=MAX_WORDS, tools=list(TOOLS),
                language='en', rubric_version='0.1', source='SpecMine local snapshot',
                roles={'speckit': ['feature'], 'openspec': ['change_proposal', 'archived']})
run_key = hashlib.sha256(json.dumps({'settings': settings, 'manifest': manifest},
                                  sort_keys=True).encode()).hexdigest()[:12]
OUTPUT = Path('output/rubric_pilot') / run_key
OUTPUT.mkdir(parents=True, exist_ok=True)
def write_once(path, content):
    try:
        with path.open('x', encoding='utf-8') as stream:
            stream.write(content)
    except FileExistsError:
        pass  # Preserve annotations and reading copies from previous runs.

write_once(OUTPUT / 'manifest.json', json.dumps({'settings': settings, 'specs': manifest}, indent=2))
rubric_text = """# Rubric pilot

Read the grading outline in example.ipynb section 8.

20 candidates are ungraded; choose 5 good and 5 bad only after human review.
Use the line-numbered reading packet for evidence. Scores: 0, 1, 2, U, NA; blank means not rated.
Keep reviewer A and B independent. Preserve original ratings and reconcile separately.
JSON files are optional machine-readable equivalents of the Markdown worksheets; edits are not synced.
Choose one format as the record of truth. Reruns preserve both formats.
The manifest records snapshot provenance, not a verified pre-implementation revision.
"""
write_once(OUTPUT / 'README.md', rubric_text)
packet = ['# Candidate specifications — full snapshot text',
          'Use S IDs and line numbers in the worksheets. No automated quality labels.']
for row in candidates.itertuples():
    content = text_by_id[row.file_url_sha16]
    numbered = '\n'.join(f'{i:04d} | {line}' for i, line in enumerate(content.splitlines(), 1))
    # Indented code preserves arbitrary Markdown fences and prevents active HTML/images.
    packet.extend([f'## {row.review_id}', '\n'.join('    ' + line for line in numbered.splitlines())])
    write_once(OUTPUT / f'{row.review_id}.md', content)
write_once(OUTPUT / 'reading_packet.md', '\n\n'.join(packet) + '\n')
for reviewer in ['A', 'B', 'consensus']:
    records, forms = [], [f'# Reviewer {reviewer} — rubric v0.1',
                          'Use 0/1/2/U/NA. Leave unreviewed scores blank. Cite lines and justify every score.']
    for row in candidates.itertuples():
        records.append(dict(review_id=row.review_id, reviewer=reviewer, rubric_version='0.1',
                            purpose='', task_type='', scope='', context_needed='', initial_label='',
                            initial_reason='', scores={d: None for d in DIMENSIONS},
                            evidence={d: '' for d in DIMENSIONS}, final_label='', final_reason='',
                            select_for_pilot=False, disagreement_notes=''))
        forms.append(f'## {row.review_id}\n\nPurpose: \nTask type / scope: \nContext needed: \n'
                     'Initial impression (good/bad/mixed/uncertain) and reason: \n\n'
                     '| Dimension | Score | Evidence (lines/quote) and reason |\n'
                     '|---|---|---|\n' +
                     '\n'.join(f'| {d} | | |' for d in DIMENSIONS) +
                     '\n\nFinal label and reason: \nSelected for 10-spec pilot? \nDisagreement notes: \n')
    write_once(OUTPUT / f'reviewer_{reviewer}.json', json.dumps(records, indent=2))
    write_once(OUTPUT / f'reviewer_{reviewer}.md', '\n\n'.join(forms) + '\n')
write_once(OUTPUT / 'rubric_changes.md', '# Rubric change log\n\n'
           '| Version | Date | Changed anchor | Evidence/reason | Cases to rescore |\n'
           '|---|---|---|---|---|\n| 0.1 | | Initial provisional rubric in notebook | Planning documents | All pilot cases |\n')
print(f'Reading packet, manifest, and blank grading forms: {OUTPUT.resolve()}')
print('Rerunning preserves existing annotations. Changed samples/settings use a separate directory.')

Reading packet, manifest, and blank grading forms: /Users/vserra/Downloads/CS520/specmine-official-main/output/rubric_pilot/1a843dcb5c37
Rerunning preserves existing annotations. Changed samples/settings use a separate directory.


### Optional: summarize completed JSON reviews

Edit reviewer A/B JSON files in the printed directory if you want this summary; Markdown-only annotations are not imported. Blank values are deliberately left ungraded. This cell validates allowed scores and required explanations before reporting agreement. Consensus labels and final pilot selection remain human decisions.

In [12]:
def load_ratings(reviewer):
    records = json.loads((OUTPUT / f'reviewer_{reviewer}.json').read_text())
    expected = set(candidates.review_id)
    if len(records) != len(expected) or {r['review_id'] for r in records} != expected:
        raise ValueError('Reviewer IDs must match the candidate manifest exactly once.')
    rows = []
    for r in records:
        for dimension in DIMENSIONS:
            score = r['scores'][dimension]
            valid = score is None or (type(score) is int and score in (0, 1, 2)) or (
                type(score) is str and score in ('U', 'NA'))
            if not valid:
                raise ValueError(f"Invalid score: {r['review_id']} / {dimension}: {score}")
            if score is not None and not r['evidence'][dimension].strip():
                raise ValueError(f"Evidence/reason required: {r['review_id']} / {dimension}")
        numeric = [r['scores'][d] for d in DIMENSIONS if type(r['scores'][d]) is int]
        rows.append({'review_id': r['review_id'], **r['scores'],
                     'total_0_to_10': sum(numeric) if len(numeric) == 5 else None})
    return pd.DataFrame(rows).set_index('review_id')
ratings_a, ratings_b = load_ratings('A'), load_ratings('B')
agreement = []
for dimension in DIMENSIONS:
    a = pd.to_numeric(ratings_a[dimension], errors='coerce')
    b = pd.to_numeric(ratings_b[dimension], errors='coerce')
    joint = a.notna() & b.notna()
    agreement.append({'dimension': dimension, 'jointly_numeric': int(joint.sum()),
                      'exact_agreement': float(a[joint].eq(b[joint]).mean()) if joint.any() else None,
                      'disagreements': list(a.index[joint & a.ne(b)])})
display(pd.DataFrame(agreement))
print('No agreement estimate is reported until reviewers supply numeric ratings.')

No agreement estimate is reported until reviewers supply numeric ratings.


,dimension,jointly_numeric,exact_agreement,disagreements
0,requirements_scope,0,None,[]
1,acceptance_criteria,0,None,[]
2,clarity_conciseness,0,None,[]
3,references_constraints,0,None,[]
4,verification_guidance,0,None,[]
